# Questionnaire Physiology

Link per-question psychometric responses to concurrent HR/IBI physiology.

**Reads:** `data/individual/ (one participant, per session)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

hr_baseline = pd.read_csv(f'{DATA}/hr.csv')
hr_01 = pd.read_csv(f'{DATA}/hr_01.csv')
hr_02 = pd.read_csv(f'{DATA}/hr_02.csv')
hr_03 = pd.read_csv(f'{DATA}/hr_03.csv')

ibi_baseline = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

eye_tracking_baseline = pd.read_csv(f'{DATA}/sed.csv')
eye_tracking_01 = pd.read_csv(f'{DATA}/sed_01.csv')
eye_tracking_02 = pd.read_csv(f'{DATA}/sed_02.csv')
eye_tracking_03 = pd.read_csv(f'{DATA}/sed_03.csv')

psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

columns_mapping = {'datetime': 'timestamp', 'pupil': 'pupil_dilation', 'leftEyeOpen': 'left_blink', 'rightEyeOpen': 'right_blink'}
eye_tracking_baseline, eye_tracking_01, eye_tracking_02, eye_tracking_03 = [
    df.rename(columns=columns_mapping) for df in
    [eye_tracking_baseline, eye_tracking_01, eye_tracking_02, eye_tracking_03]
]

def clean_eye(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce').dt.tz_convert(None)
    return df.dropna(subset=['timestamp'])

def clean_hrv(df):
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    return df.dropna(subset=['datetime'])

eye_tracking_baseline = clean_eye(eye_tracking_baseline)
eye_tracking_01 = clean_eye(eye_tracking_01)
eye_tracking_02 = clean_eye(eye_tracking_02)
eye_tracking_03 = clean_eye(eye_tracking_03)

hr_baseline = clean_hrv(hr_baseline)
hr_01 = clean_hrv(hr_01)
hr_02 = clean_hrv(hr_02)
hr_03 = clean_hrv(hr_03)
ibi_baseline = clean_hrv(ibi_baseline)
ibi_01 = clean_hrv(ibi_01)
ibi_02 = clean_hrv(ibi_02)
ibi_03 = clean_hrv(ibi_03)

psychometric_01['Question Start Time'] = pd.to_datetime(psychometric_01['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Start Time'] = pd.to_datetime(psychometric_02['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Start Time'] = pd.to_datetime(psychometric_03['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_01['Question Answer Time'] = pd.to_datetime(psychometric_01['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Answer Time'] = pd.to_datetime(psychometric_02['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Answer Time'] = pd.to_datetime(psychometric_03['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

def calculate_hrv_metrics(ibi_data):
    if len(ibi_data) == 0:
        return None, None
    valid_ibi = ibi_data[ibi_data > 0]
    rmssd = np.sqrt(np.mean(np.square(np.diff(valid_ibi))))
    sdnn = np.std(valid_ibi, ddof=1)
    return rmssd, sdnn

def calculate_blink_rate(blink_data, duration_s):
    if duration_s <= 0:
        return 0.0
    blink_count = ((blink_data > 1.0) & (blink_data.shift(-1) <= 1.0)).sum()
    return blink_count / (duration_s / 60)

baseline_avg_hr = hr_baseline['heart_rate'].mean()
baseline_duration_s = (eye_tracking_baseline['timestamp'].max() - eye_tracking_baseline['timestamp'].min()).total_seconds()

baseline_metrics = {
    'rmssd': calculate_hrv_metrics(ibi_baseline['ibi'])[0],
    'sdnn': calculate_hrv_metrics(ibi_baseline['ibi'])[1],
    'pupil_dilation': eye_tracking_baseline['pupil_dilation'].mean(),
    'left_blink_rate': calculate_blink_rate(eye_tracking_baseline['left_blink'], baseline_duration_s),
    'right_blink_rate': calculate_blink_rate(eye_tracking_baseline['right_blink'], baseline_duration_s),
}

def process_question_data(questions, hr_data, ibi_data, eye_tracking_data, baseline_avg_hr, baseline_metrics):
    results = []
    for _, question in questions.iterrows():
        start_time = question['Question Start Time']
        end_time = question['Question Answer Time']
        duration_s = (end_time - start_time).total_seconds()

        hr_segment = hr_data[(hr_data['datetime'] >= start_time) & (hr_data['datetime'] <= end_time)]
        ibi_segment = ibi_data[(ibi_data['datetime'] >= start_time) & (ibi_data['datetime'] <= end_time)]
        eye_segment = eye_tracking_data[(eye_tracking_data['timestamp'] >= start_time) & (eye_tracking_data['timestamp'] <= end_time)]

        avg_hr = hr_segment['heart_rate'].mean()
        rmssd, sdnn = calculate_hrv_metrics(ibi_segment['ibi'])
        avg_pupil = eye_segment['pupil_dilation'].mean()
        left_blink = calculate_blink_rate(eye_segment['left_blink'], duration_s)
        right_blink = calculate_blink_rate(eye_segment['right_blink'], duration_s)

        results.append({
            'Question': question['Question'],
            'Start Time': start_time.strftime('%H:%M:%S'),
            'End Time': end_time.strftime('%H:%M:%S'),
            'Duration (s)': duration_s,
            'Average HR (BPM)': avg_hr,
            'RMSSD (ms)': rmssd,
            'SDNN (ms)': sdnn,
            'Average Pupil Dilation': avg_pupil,
            'Left Blink Rate': left_blink,
            'Right Blink Rate': right_blink,
        })
    return results

def display_results(results, test_number):
    print(f"\nTest {test_number:02d}:")
    for r in results:
        print(f"  {r['Question']}: {r['Start Time']} - {r['End Time']} ({r['Duration (s)']}s)")
        hr_val = r['Average HR (BPM)']
        rmssd_val = r['RMSSD (ms)']
        sdnn_val = r['SDNN (ms)']
        print(f"    HR: {hr_val:.2f} BPM" if hr_val is not None else "    HR: N/A")
        print(f"    RMSSD: {rmssd_val:.2f} ms  SDNN: {sdnn_val:.2f} ms" if rmssd_val is not None else "    RMSSD: N/A  SDNN: N/A")
        print(f"    Pupil: {r['Average Pupil Dilation']:.2f}  Left blink: {r['Left Blink Rate']:.2f}  Right blink: {r['Right Blink Rate']:.2f}")

sessions = [
    (psychometric_01, hr_01, ibi_01, eye_tracking_01),
    (psychometric_02, hr_02, ibi_02, eye_tracking_02),
    (psychometric_03, hr_03, ibi_03, eye_tracking_03),
]

for i, (psy, hr, ibi, eye) in enumerate(sessions, 1):
    results = process_question_data(psy[psy['Type'] == 'HADS'], hr, ibi, eye, baseline_avg_hr, baseline_metrics)
    display_results(results, i)

In [ ]:
import pandas as pd
import numpy as np

PSY = '../data/individual/psychometric'

# load psychometric data
psy_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psy_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psy_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

# clinical thresholds
thresholds = {
    'HADS': {'normal': 7, 'borderline': 10, 'label': '0-7 normal, 8-10 borderline, 11+ abnormal'},
    'STAI-S': {'normal': 39, 'borderline': 54, 'label': '20-39 low, 40-54 moderate, 55+ high'},
    'STAI-T': {'normal': 39, 'borderline': 54, 'label': '20-39 low, 40-54 moderate, 55+ high'},
}

# total scores
print("=== Psychometric Total Scores ===\n")
for i, psy in enumerate([psy_01, psy_02, psy_03], 1):
    print(f"--- Session {i:02d} ---")
    for q_type in psy['Type'].unique():
        subset = psy[psy['Type'] == q_type]
        answers = pd.to_numeric(subset['Answer'], errors='coerce')
        total = answers.sum()
        n_items = len(answers.dropna())
        mean_score = answers.mean()

        if q_type in thresholds:
            t = thresholds[q_type]
            if total <= t['normal']:
                level = "normal"
            elif total <= t['borderline']:
                level = "borderline"
            else:
                level = "elevated"
            print(f"  {q_type}: total={total}, mean={mean_score:.2f}, n={n_items} -> {level} ({t['label']})")
        else:
            print(f"  {q_type}: total={total}, mean={mean_score:.2f}, n={n_items}")
    print()

# session comparison
print("=== Score Changes Across Sessions ===\n")
for q_type in psy_01['Type'].unique():
    scores = []
    for psy in [psy_01, psy_02, psy_03]:
        scores.append(pd.to_numeric(psy[psy['Type'] == q_type]['Answer'], errors='coerce').sum())
    trend = "increasing" if scores[-1] > scores[0] else "decreasing" if scores[-1] < scores[0] else "stable"
    print(f"{q_type}: {scores} ({trend})")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PSY = '../data/individual/psychometric'

# load data
psy_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psy_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psy_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

# extract item number
def get_item_num(test_str):
    part = str(test_str).split('.')[-1]
    return int(part) if part.isdigit() else 0

# --- HADS subscale separation ---
print("=== HADS Subscale Analysis ===\n")
print("Anxiety items: 1,3,5,7,9,11,13 | Depression items: 2,4,6,8,10,12,14")
print("Threshold: 0-7 normal, 8-10 borderline, 11+ clinical\n")

for i, psy in enumerate([psy_01, psy_02, psy_03], 1):
    hads = psy[psy['Type'] == 'HADS'].copy()
    hads['item_num'] = hads['Test'].apply(get_item_num)
    
    anxiety_items = hads[hads['item_num'].isin([1, 3, 5, 7, 9, 11, 13])]
    depression_items = hads[hads['item_num'].isin([2, 4, 6, 8, 10, 12, 14])]
    
    anx_total = pd.to_numeric(anxiety_items['Answer'], errors='coerce').sum()
    dep_total = pd.to_numeric(depression_items['Answer'], errors='coerce').sum()
    
    anx_level = "normal" if anx_total <= 7 else "borderline" if anx_total <= 10 else "clinical"
    dep_level = "normal" if dep_total <= 7 else "borderline" if dep_total <= 10 else "clinical"
    
    print(f"Session {i:02d}: Anxiety={anx_total} ({anx_level}), Depression={dep_total} ({dep_level})")

# --- session trajectories ---
print("\n=== Subscale Trajectories ===\n")
subscale_scores = {'HADS-A': [], 'HADS-D': [], 'STAI-S': [], 'STAI-T': []}

for psy in [psy_01, psy_02, psy_03]:
    hads = psy[psy['Type'] == 'HADS'].copy()
    hads['item_num'] = hads['Test'].apply(get_item_num)
    subscale_scores['HADS-A'].append(pd.to_numeric(hads[hads['item_num'].isin([1, 3, 5, 7, 9, 11, 13])]['Answer'], errors='coerce').sum())
    subscale_scores['HADS-D'].append(pd.to_numeric(hads[hads['item_num'].isin([2, 4, 6, 8, 10, 12, 14])]['Answer'], errors='coerce').sum())
    subscale_scores['STAI-S'].append(pd.to_numeric(psy[psy['Type'] == 'STAI-S']['Answer'], errors='coerce').sum())
    subscale_scores['STAI-T'].append(pd.to_numeric(psy[psy['Type'] == 'STAI-T']['Answer'], errors='coerce').sum())

for name, scores in subscale_scores.items():
    trend = "increasing" if scores[-1] > scores[0] else "decreasing" if scores[-1] < scores[0] else "stable"
    print(f"{name}: {scores} ({trend})")

# --- radar chart ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(polar=True))
categories = list(subscale_scores.keys())
n_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

# normalize for radar
max_vals = {'HADS-A': 21, 'HADS-D': 21, 'STAI-S': 80, 'STAI-T': 80}
colors = ['tab:blue', 'tab:orange', 'tab:green']

for idx in range(3):
    ax = axes[idx]
    values = [subscale_scores[cat][idx] / max_vals[cat] * 100 for cat in categories]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', color=colors[idx], linewidth=2)
    ax.fill(angles, values, alpha=0.25, color=colors[idx])
    ax.set_thetagrids(np.degrees(angles[:-1]), categories)
    ax.set_ylim(0, 100)
    ax.set_title(f'Session {idx+1:02d}', pad=20)
    ax.set_yticks([25, 50, 75])
    ax.set_yticklabels(['25%', '50%', '75%'], fontsize=7)

plt.suptitle('Psychometric Profile (% of max score)', y=1.05)
plt.tight_layout()
plt.show()
plt.close()

# --- trajectory line plot ---
fig, ax = plt.subplots(figsize=(10, 6))
sessions = ['Session 01', 'Session 02', 'Session 03']
for name, scores in subscale_scores.items():
    normalized = [s / max_vals[name] * 100 for s in scores]
    ax.plot(sessions, normalized, 'o-', label=name, linewidth=2, markersize=8)

ax.axhline(50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.set_ylabel('Score (% of maximum)')
ax.set_title('Psychometric Subscale Trajectories Across Sessions')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
plt.close()